In [14]:
import os

import pandas as pd
import geopandas as gpd
from tqdm import tqdm

In [7]:
# CELL 2 — Load the Geo Starting Row Index and Filter Toronto ADAs

geo_index = pd.read_csv("../data/ada/98-401-X2021012_Geo_starting_row.CSV",
                        dtype=str)

geo_index["Line Number"] = geo_index["Line Number"].astype(int)

# Region prefixes we want
prefixes = [
    # "3525",  # Hamilton
    # "3524",  # Halton
    # "3521",  # Peel
    # "3519",  # York
    "3520",  # Toronto
    # "3518",  # Durham
]  

mask = geo_index["Geo Name"].str.startswith(tuple(prefixes))
toronto_geo = geo_index[mask].copy()

print(f"Toronto ADA count: {len(toronto_geo)}")

ontario_geo = toronto_geo

Toronto ADA count: 279


In [8]:
# Sort ADAs by row order
ontario_geo = ontario_geo.sort_values("Line Number").reset_index(drop=True)

# Compute row ranges (start row, end row)
start_rows = ontario_geo["Line Number"].astype(int).tolist()
end_rows = start_rows[1:] + [None]

row_ranges = list(zip(start_rows, end_rows))

row_ranges[:5]
# BUG: for some reason the last range is to the EOF?

[(5264633, 5267264),
 (5267264, 5269895),
 (5269895, 5272526),
 (5272526, 5275157),
 (5275157, 5277788)]

In [11]:
# CELL 4 — Extract each ADA block individually, fix simple off-by-one, filter IDs & columns

main_file = "../data/ada/98-401-X2021012_English_CSV_data.csv"
output_dir = "../data/ada/toronto"
os.makedirs(output_dir, exist_ok=True)

print("Extracting each Toronto ADA block one by one with filtering...")

# Define CHARACTERISTIC_ID ranges
id_ranges = [
    (1, 1),
    (2, 2),
    (3, 3),
    (34, 40),
    (111, 125),
    (155, 186),
    (260, 300),
    (345, 349),
    (365, 377),
    (383, 387),
    (724, 734),
    (1414, 1418),
    (1522, 1526),
    (1683, 1697),
    (1998, 2001),
    (2228, 2230),
    (2246, 2281),
]

# Flatten ranges into a set
valid_ids = set()
for start, end in id_ranges:
    valid_ids.update(range(start, end + 1))

required_cols = [
    "GEO_NAME",
    "CHARACTERISTIC_ID",
    "CHARACTERISTIC_NAME",
    "C1_COUNT_TOTAL",
    "C10_RATE_TOTAL"
]

for i, (start, end) in enumerate(tqdm(row_ranges)):
    ada_id = ontario_geo.iloc[i]["Geo Name"]  # e.g., 35200001
    output_file = os.path.join(output_dir, f"{ada_id}.csv")

    # FIX: skip all rows BEFORE the ADA block (subtract 1 for zero-based CSV)
    skip = list(range(1, start - 1))  # leave row 'start' as first row read

    # Determine block length
    if end is not None:
        nrows = end - start
    else:
        nrows = None  # last ADA goes to EOF

    # Read block
    df_block = pd.read_csv(
        main_file,
        skiprows=skip,
        nrows=nrows,
        low_memory=False,
        encoding='latin'
    )

    # Filter CHARACTERISTIC_ID
    df_block["CHARACTERISTIC_ID"] = pd.to_numeric(df_block["CHARACTERISTIC_ID"], errors='coerce')
    df_block = df_block[df_block["CHARACTERISTIC_ID"].isin(valid_ids)]

    # Keep only required columns
    df_block = df_block[required_cols]

    # Save CSV
    df_block.to_csv(output_file, index=False)

Extracting each Toronto ADA block one by one with filtering...


100%|██████████| 279/279 [21:32<00:00,  4.63s/it]


In [16]:
# CELL 5 — Merge all Toronto ADA CSVs into one final CSV

output_dir = "../data/ada/toronto"
final_output = "../data/ada/toronto-ada.csv"

csv_files = sorted(os.listdir(output_dir))
write_header = True  # write header only for first file

with open(final_output, "w", encoding="latin") as outfile:
    for fname in tqdm(csv_files):
        fpath = os.path.join(output_dir, fname)
        with open(fpath, "r", encoding="latin") as infile:
            if write_header:
                outfile.write(infile.read())
                write_header = False
            else:
                next(infile)  # skip header
                outfile.write(infile.read())

print("Final merged CSV saved:")
print(final_output)

100%|██████████| 279/279 [00:00<00:00, 6230.66it/s]

Final merged CSV saved:
../data/ada/toronto-ada.csv


In [15]:
# CELL 6 — Load ADA shapefile, filter for Toronto, save as GeoPackage

# File path for the full Canada ADA shapefile
shapefile_path = "../data/ada/lada000b21a_e"

# Load shapefile
gdf = gpd.read_file(shapefile_path)

# Filter for Toronto ADAs using ADAUID prefix
toronto_prefixes = ["3520"]  # same as Cell 2
gdf_toronto = gdf[gdf["ADAUID"].str.startswith(tuple(toronto_prefixes))].copy()

# Save filtered GeoDataFrame to GeoPackage
output_gpkg = "../data/ada/toronto-ada.gpkg"
gdf_toronto.to_file(output_gpkg, driver="GPKG")

print(f"Saved Toronto ADA geometries to: {output_gpkg}")
print(f"Number of Toronto ADAs: {len(gdf_toronto)}")


Saved Toronto ADA geometries to: ../data/ada/toronto-ada.gpkg
Number of Toronto ADAs: 279
